# 05 — Geometry on Real Hardware: the Bures Distance

How far did the device's state fall from the ideal you asked for? In `05/03` you answered with a fidelity below 1 — an honest score, but a *number*, not yet a *distance*. Here you turn it into one. The gap becomes a single point on the Bures geometry you built in `02/12` — the quantum Fisher–Rao distance — and then you watch that point travel outward as you add circuit depth, decoherence pushing the device's state steadily further from the ideal.

**Prerequisites:** `02/12_bures_distance`, `05/03_tomography_and_fidelity_on_real_hardware`, `05/04_quantum_mutual_information_on_real_hardware`.

**What you'll be able to do**
- Reconstruct a device state exactly as in `05/03` and measure `bures_distance(rho_hat, ideal)` — the decoherence gap read off the Bures geometry as a single number.
- Dial decoherence up with identity padding and watch the Bures distance rise while fidelity falls — two views of one physical effect.
- Place the device on the very geometry that runs through modules 02–04, the geometry that powers quantum optimal transport.

You already have every piece. `05/03` reconstructed `rho_hat` from a device's clicks and scored its fidelity; `02/12` built the Bures distance as the quantum analogue of the Fisher–Rao metric. Here those two threads meet: you feed the device's reconstructed state straight into the Bures distance and read the decoherence gap as a geometric length. Then, by lengthening the circuit, you make that length grow before your eyes — error you can *see moving* on a geometry. Let us go and measure it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import single_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import bures_distance
from qot_course.quantum.density import density_matrix, fidelity
from qot_course.quantum.states import KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

## 1 · A geometric read-out of decoherence

In `05/03` the fidelity told you *how close* the device's state landed to the ideal, on a scale from 0 to 1. The Bures distance turns that closeness into a genuine length. For two states $\hat\rho$ and the ideal target it is

$$d_B(\hat\rho, \mathrm{ideal}) = \sqrt{2\,\bigl(1 - \sqrt{F}\bigr)},$$

where $F$ is the fidelity you already know. This is the quantum Fisher–Rao distance you built in `02/12`: the geodesic length between two states on the Bures geometry, equal to 0 exactly when the states are identical and growing as they separate. It is the quantum sibling of the classical Fisher–Rao distance from `02/06` — the same information-geometric idea, lifted from probability vectors to density matrices.

The plan is direct. We reconstruct $\hat\rho$ exactly as in `05/03` — single-qubit Pauli tomography through `SamplerV2`, reassembled by `density_from_counts` — and hand it to `bures_distance`. The decoherence gap that `05/03` reported as a fidelity now reads as a *distance*: one number locating the device on the geometry.

## 2 · Dialing decoherence

A single reading is one point. To watch the geometry *respond*, we need a knob that adds error in controlled amounts — and the cleanest knob is circuit depth that leaves the target untouched.

We insert $k$ identity layers before tomography. Each layer is an `x` followed by another `x`: $X \cdot X = I$, so the state we *intend* to prepare never changes — it stays $|+\rangle$ for every $k$. What does change is the number of physical gates the device must execute, and every real gate leaks a little error. More depth means more accumulated gate error, so the reconstructed $\hat\rho$ drifts further from the ideal and the Bures distance grows.

There is one subtlety that makes or breaks the experiment. A transpiler is built to *simplify* circuits, and an $X \cdot X$ pair is exactly the kind of redundancy it loves to cancel — left alone, it would erase every padding gate and the sweep would go flat. To stop it, we wrap each `x` in barriers: `prep.barrier(); prep.x(0); prep.barrier(); prep.x(0)`. A barrier forbids the transpiler from moving gates across it, so the pairs survive transpilation and actually run on the device. Without the barriers there is no decoherence to measure; with them, the error accumulates as intended.

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

if not is_real:
    backend.set_options(seed_simulator=42)  # reproducible offline figure; a real QPU has no such option
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
ideal = density_matrix(KET_PLUS)
k_values = [0, 4, 8, 16]
bures, fids = [], []
for k in k_values:
    prep = QuantumCircuit(1)
    prep.h(0)
    for _ in range(k):                       # identity padding (X X = I); barriers stop the transpiler cancelling it
        prep.barrier(); prep.x(0); prep.barrier(); prep.x(0)
    circuits = single_qubit_tomography_circuits(prep)
    settings = list(circuits)
    isa_list = [pm.run(circuits[s]) for s in settings]
    res = SamplerV2(mode=backend).run(isa_list, shots=8192).result()
    cbs = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}
    rho = density_from_counts(cbs, n_qubits=1)
    bures.append(bures_distance(rho, ideal))
    fids.append(fidelity(rho, ideal))
for k, d, f in zip(k_values, bures, fids):
    print(f"padding k={k:2d} ({2 * k} X gates): Bures = {d:.4f}, fidelity = {f:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(k_values, bures, "o-", color=COLORS["quantum"])
ax1.set_xlabel("identity-padding depth k"); ax1.set_ylabel("Bures distance to |+>")
ax1.set_title("Bures distance grows with decoherence", pad=10)
ax2.plot(k_values, fids, "s-", color=COLORS["flow"])
ax2.set_xlabel("identity-padding depth k"); ax2.set_ylabel("fidelity to |+>")
ax2.set_title("...as fidelity falls", pad=10)
fig.tight_layout()
plt.show()

**Read the figure.** Two views of one physical effect. On the left the Bures distance rises monotonically with padding depth; on the right the fidelity falls — the same decoherence, seen once as a geometric length and once as an overlap. The k=0 baseline is not zero, and that honesty matters: it is the Hadamard-prep error plus the read-out error, together with finite-shot and projection bias from reconstructing $\hat\rho$ from a finite number of clicks. The *rise* across the sweep is the part that is cleanly attributable to the added `x` gates — each padding layer is two more chances to err. The effect per layer is modest but monotonic: a single qubit idling under a few extra gates drifts only a little, yet it drifts *outward* every time. (Offline this reflects the modeled FakeManila noise; flip `USE_HARDWARE=True` and the very same curves report the literal chip.)

### A real run on IBM hardware

On **`ibm_marrakesh`** (2026-05-30), the same depth sweep gave Bures distances of about **0.116, 0.127, 0.136, 0.125** for k = 0, 4, 8, 16 (fidelity about 0.98 throughout). On this modern processor the single-qubit gates are clean enough that 32 identity `X` gates barely move the state — the four points sit within shot-noise of one another, so the crisp monotonic rise the *noisier* FakeManila model shows is, here, **below the noise floor**. Resolving the trend cleanly on such a device would take many more shots, or a deliberately noisier qubit. (The offline figure above uses the FakeManila model precisely so the principle — more decoherence, larger Bures — stays visible.) Set `USE_HARDWARE = True` to reproduce.

## 3 · The geometry behind this distance

Step back and notice what this distance *is*. The Bures geometry you measured a device on is the same one you built in `02/12`, the same one bridged to the Wasserstein distance of optimal transport in `03/12–13`, and the same one underlying the quantum optimal-transport measures of module 04. The geometry that powers quantum optimal transport is exactly the geometry that quantified a real chip's error a moment ago. The device's decoherence and the mathematics of QOT are not two subjects — they are one geometry, met from two directions.

## Your turn

- **Swap the padding for a `delay`.** If your backend supports the `delay` instruction, replace the `x·x` layers with an idle `delay` of growing duration and compare the curves. Gate error and idle decoherence (T1/T2) are different physical channels — does pure idling move the Bures distance the same way the gates do?
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The distances you read are now the real chip's drift along the Bures geometry, not a model's.
- **Check the closed form.** For each k, compute `np.sqrt(2 * (1 - np.sqrt(f)))` from the printed fidelity `f` and compare it to the printed Bures value. They should match to all printed digits — the same number, once read off the device and once off the formula of `02/12`.

## What you built

- A geometric read-out of decoherence: you reconstructed a device state with single-qubit tomography and measured `bures_distance(rho_hat, ideal)`, turning the fidelity gap of `05/03` into a genuine distance on the Bures geometry.
- A controlled decoherence sweep: identity padding (with barriers to survive transpilation) that lengthens the circuit while leaving the target fixed, so you could watch the Bures distance rise monotonically and the fidelity fall in step.
- An honest decomposition of the gap: a nonzero k=0 baseline (prep + read-out + finite-shot/projection bias) and a clean *rise* attributable to the added gate error.
- The closing of the module's geometric arc: the device placed on the same Bures geometry that runs through modules 02–04 and powers quantum optimal transport.

## What's next

Next is the **capstone** of the real-hardware module: you bring tomography, fidelity, mutual information, and the Bures distance together on a live device, and look out toward the frontier of where quantum optimal transport meets real quantum machines.

## References

- D. Bures, "An extension of Kakutani's theorem on infinite product measures to the tensor product of semifinite $w^*$-algebras," *Trans. Amer. Math. Soc.* **135**, 199 (1969).
- M. M. Wilde, *Quantum Information Theory*, ch. 9, Cambridge University Press (2017), doi:10.1017/9781316809976.
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 9 (fidelity and distance measures), Cambridge University Press (2010).

**Previous:** `notebooks/05_RealQuantumHardware/04_quantum_mutual_information_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/06_capstone_on_real_hardware_and_frontier.ipynb`